In [1]:
import pandas as pd
import numpy as np
import cpi
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# # %pip install cpi
# cpi.update()

## IMPORTAR DATASETS

In [2]:
# Load Dataframes
print("Loading raw data...")

DATA_FILE_FACTS = 'data/raw_facts_500.pkl'
DATA_FILE_SONGS = 'data/raw_songs_500.pkl'
DATA_FILE_MOVIES = 'data/raw_movies_500.pkl'
DATA_FILE_ARTISTS = 'data/raw_artists_500.pkl'

df_facts_raw = pd.read_pickle(DATA_FILE_FACTS)
df_songs_raw = pd.read_pickle(DATA_FILE_SONGS)
df_movies_raw = pd.read_pickle(DATA_FILE_MOVIES)
df_artists_raw = pd.read_pickle(DATA_FILE_ARTISTS)

print("Data loaded! Ready for transformations...")
display(df_facts_raw.head())

Loading raw data...
Data loaded! Ready for transformations...


,song_id,year,song_title,film_title,film_id
0,193401,1934,The Continental,The Gay Divorcee,193401
1,193402,1934,Love In Bloom,She Loves Me Not,193402
2,193501,1935,Lullaby Of Broadway,Gold Diggers of 1935,193501
3,193502,1935,Cheek To Cheek,Top Hat,193502
4,193503,1935,Lovely To Look At,Roberta,193503


### Inspección de los Datasets

In [3]:
print(df_songs_raw.columns)
# display(df_songs_raw[['song_id', 'song_title', 'oscar_song_nominee', 'oscar_song_win']].head())

print(df_movies_raw.columns)
# display(df_movies_raw[['film_id', 'film_title', 'tmdb_budget', 'tmdb_revenue', 'omdb_awards']].head())

print(df_artists_raw.columns)
# display(df_artists_raw[['artist_id', 'artist_name', 'artist_playcount', 'artist_listeners']].head())

Index(['song_id', 'year', 'song_title', 'song_composers', 'song_lyricists',
       'original_artists', 'streams_original', 'other_artists',
       'streams_others', 'oscar_song_nominee', 'oscar_song_win',
       'grammy_song_nominee', 'grammy_song_win', 'grammy_record_nominee',
       'grammy_record_win', 'spotify_id', 'spotify_track_title',
       'spotify_artists', 'spotify_album_title', 'spotify_release_date',
       'spotify_duration_ms', 'isrc', 'lastfm_track_title',
       'lastfm_artist_name', 'lastfm_track_playcount',
       'lastfm_track_listeners', 'lastfm_tags', 'lastfm_track_url',
       'lastfm_duration_ms', 'track_mbid', 'artist_mbid', 'lastfm_album_title',
       'lastfm_album_artist', 'wiki_summary', 'mb_work_id', 'mb_work_title',
       'mb_iswcs', 'mb_composers', 'mb_lyricists', 'mb_total_recordings',
       'mb_original_recordings_count', 'mb_cover_recordings_count',
       'mb_covers_performers_count', 'mb_covers_performers_top',
       'mb_covers_earliest_year', 'm

### Dataframes Auxiliares

In [4]:
# Songs Dataframe
df_songs_aux = pd.DataFrame()

columns_to_keep = [
    'song_id', 'song_title', 'year',
    'lastfm_track_playcount', 'lastfm_track_listeners',
    'oscar_song_nominee', 'oscar_song_win',
    'grammy_song_nominee', 'grammy_song_win',
    'grammy_record_nominee', 'grammy_record_win',
    'song_composers', 'song_lyricists',
    'original_artists', 'streams_original', 
    'other_artists', 'streams_others', 
    'spotify_release_date', 'spotify_duration_ms',
    # 'spotify_id', 'isrc',
    'spotify_track_title', 'spotify_artists', 'spotify_album_title', 
    'lastfm_track_title', 'lastfm_artist_name', 'lastfm_album_title', 
    # 'lastfm_album_artist', 'lastfm_track_url', 'lastfm_duration_ms', 
    # 'track_mbid', 'artist_mbid', 
    'lastfm_tags', 'wiki_summary', 
    # 'mb_work_id', 'mb_iswcs',
    'mb_work_title', 'mb_composers', 'mb_lyricists', 
    'mb_total_recordings', 'mb_original_recordings_count', 'mb_cover_recordings_count',
    'mb_covers_performers_count', 'mb_covers_performers_top',
    'mb_covers_earliest_year', 'mb_covers_earliest_performer',
    'mb_covers_latest_year', 'mb_covers_latest_performer', 
    'am_ranking', 'am_song_artist', 'am_song_year', 
    'cm_ranking', 'cm_song_artist', 'cm_song_eas',
    'match_key_am', 'match_key_cm'
]

df_songs_aux = df_songs_raw[columns_to_keep].copy()

# Films Dataframe
df_films_aux = pd.DataFrame()

columns_to_keep = [
    'film_id', 'year', 'film_title', 'film_directors', 'film_composers',
    'oscar_nominations', 'oscar_wins', 
    # 'tmdb_id', 'imdb_id',
    'tmdb_official_title', 'tmdb_original_title', 'tmdb_release_date',
    'tmdb_runtime', 'tmdb_budget', 'tmdb_revenue', 'tmdb_vote_average',
    'tmdb_vote_count', 'tmdb_original_language', 'tmdb_origin_countries',
    'tmdb_actors', 'tmdb_directors', 'tmdb_composers',
    'tmdb_production_companies', 'tmdb_genres', 'tmdb_tagline',
    'omdb_title', 'omdb_awards', 'omdb_imdb_rating', 'omdb_imdb_votes',
    'omdb_metascore', 'omdb_rotten_tomatoes'
]

df_films_aux = df_movies_raw[columns_to_keep].copy()

# Artists Dataframe
columns_to_keep = [
    'artist_id', 'artist_name', 'roles', # 'mb_artist_id', 
    'artist_playcount', 'artist_listeners', 
    'artist_bio', 'mb_artist_type', 'mb_gender',
    'mb_country', 'mb_begin_area', 'mb_disambiguation', 
    'mb_lifespan_begin', 'mb_lifespan_end', 'mb_is_active', # 'mb_isnis', 
    'mb_aliases', 'mb_tags', # 'mb_wikidata_url', 'mb_imdb_url', 
    'cm_artist_ranking', 'cm_artist_eas'
]

df_artists_aux = df_artists_raw[columns_to_keep].copy()

### Comprobaciones de Calidad

In [5]:
"""
df_songs_aux[['spotify_artists', 'lastfm_artist_name', 
              'streams_original', 'streams_others', 
              'lastfm_track_playcount','lastfm_track_listeners', 
              'mb_total_recordings', 'mb_original_recordings_count',
              'mb_cover_recordings_count', 'mb_covers_performers_count',
              'am_ranking', 'am_song_artist', 'cm_ranking', 'cm_song_artist', 
              'cm_song_eas']].info()
"""

# Display Missing Artists
missing_mask = df_songs_aux['spotify_artists'].isna() | df_songs_aux['lastfm_artist_name'].isna()

cols_to_view = [
    'song_id', 'song_title', 
    'original_artists', 'other_artists',
    'spotify_artists', 'lastfm_artist_name',
    'streams_original', 'streams_others'
]

# Apply the filter and view the results
df_missing = df_songs_aux.loc[missing_mask, cols_to_view]
# print(df_missing.shape)
display(df_missing)

,song_id,song_title,original_artists,other_artists,spotify_artists,lastfm_artist_name,streams_original,streams_others
25,194003,Down Argentina Way,"Betty Grable, Don Ameche",NaN,NaN,NaN,12700.0,NaN
133,196304,So Little Time,Andy Williams,NaN,NaN,NaN,189900.0,NaN
198,197403,Benji's Theme (I Feel Love),Euel Box,NaN,[Euel Box],NaN,30500.0,NaN
257,198304,Over You,Betty Buckley,NaN,NaN,NaN,NaN,NaN
285,198605,Mean Green Mother from Outer Space,Levi Stubbs,NaN,NaN,NaN,5905000.0,NaN
311,199001,Sooner Or Later (I Always Get My Man),Madonna,NaN,NaN,NaN,2190000.0,NaN
315,199005,I'm Checkin' Out,"Meryl Streep, Blue Rodeo",NaN,NaN,NaN,NaN,NaN
330,199304,The Day I Fall In Love,"Dolly Parton, James Ingram",NaN,NaN,NaN,17919000.0,NaN
337,199405,Look What Love Has Done,Patty Smyth,NaN,NaN,NaN,NaN,NaN
342,199505,Moonlight,Michael Dees,NaN,NaN,NaN,NaN,NaN


In [6]:
# Display Missing Streams
missing_mask = df_songs_aux['streams_original'].isna() & df_songs_aux['streams_others'].isna()

cols_to_view = [
    'song_id', 'song_title', 'original_artists',
    'spotify_artists', 'lastfm_artist_name',
    'streams_original', 'streams_others',
    'lastfm_track_playcount','lastfm_track_listeners'
]

# Apply the filter and view the results
df_missing = df_songs_aux.loc[missing_mask, cols_to_view]
display(df_missing)

,song_id,song_title,original_artists,spotify_artists,lastfm_artist_name,streams_original,streams_others,lastfm_track_playcount,lastfm_track_listeners
251,198204,If We Were In Love,"Luciano Pavarotti, John Williams","[John Williams, Boston Pops Orchestra]",Luciano Pavarotti,NaN,NaN,762.0,207.0
257,198304,Over You,Betty Buckley,NaN,NaN,NaN,NaN,NaN,NaN
315,199005,I'm Checkin' Out,"Meryl Streep, Blue Rodeo",NaN,NaN,NaN,NaN,NaN,NaN
337,199405,Look What Love Has Done,Patty Smyth,NaN,NaN,NaN,NaN,NaN,NaN
342,199505,Moonlight,Michael Dees,NaN,NaN,NaN,NaN,NaN,NaN
396,200405,In The Deep,Bird York,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Acclaimed Music matched songs quality check
mask = df_songs_aux['am_ranking'].notna()

cols_to_view = [
    'song_id', 'original_artists',
    'am_song_artist', 'year', 'am_song_year'
]

df_aux = df_songs_aux.loc[mask, cols_to_view]
print(df_aux.to_string())

     song_id                   original_artists                     am_song_artist  year  am_song_year
5     193601                       Fred Astaire                       Fred Astaire  1936          1936
7     193603                        Bing Crosby                        Bing Crosby  1936          1936
11    193702                       Fred Astaire                       Fred Astaire  1937          1937
21    193901                       Judy Garland                       Judy Garland  1939          1939
23    194001                      Cliff Edwards                      Cliff Edwards  1940          1940
33    194103                       Glenn Miller                       Glenn Miller  1941          1941
36    194106                The Andrews Sisters                The Andrews Sisters  1941          1941
39    194201                        Bing Crosby                        Bing Crosby  1942          1942
48    194401                        Bing Crosby                        Bi

In [8]:
# Chart Masters matched songs quality check
mask = df_songs_aux['cm_ranking'].notna()

cols_to_view = [
    'song_id', 'original_artists', 'song_title',
    # 'am_song_artist', 'year', 'am_song_year'
    'match_key_cm', 'cm_song_eas' # 'cm_ranking',
]

df_aux = df_songs_aux.loc[mask, cols_to_view]
print(df_aux.to_string())

     song_id            original_artists                                               song_title                                    match_key_cm  cm_song_eas
39    194201                 Bing Crosby                                          White Christmas                     bing crosby white christmas     19602000
48    194401                 Bing Crosby                                       Swinging On A Star                  bing crosby swinging on a star       624000
54    194407    Bing Crosby, Sonny Tufts                             Ac-Cen-Tchu-Ate the Positive          bing crosby accenttchuate the positive       758000
57    194502               Frank Sinatra                                I Fall In Love Too Easily         frank sinatra i fall in love too easily       321000
86    195401               Frank Sinatra                              Three Coins In The Fountain       frank sinatra three coins in the fountain       792000
93    195503               Frank Sinatra      

In [9]:
# Extracted TMDB and OMDB films quality check
cols_to_view = [
    'film_id', 'film_title', 'tmdb_official_title', # 'tmdb_original_title', 
    'year', 'tmdb_release_date',
    'omdb_title'
]

# Apply the filter and view the results
df_aux = df_films_aux[cols_to_view]
print(df_aux.to_string())

     film_id                                         film_title                                     tmdb_official_title  year tmdb_release_date                                               omdb_title
0     193401                                   The Gay Divorcee                                        The Gay Divorcee  1934        1934-10-12                                         The Gay Divorcee
1     193402                                   She Loves Me Not                                        She Loves Me Not  1934        1934-08-31                                         She Loves Me Not
2     193501                               Gold Diggers of 1935                                    Gold Diggers of 1935  1935        1935-03-15                                     Gold Diggers of 1935
3     193502                                            Top Hat                                                 Top Hat  1935        1935-08-29                                                  Top

In [10]:
# Display Missing Budgets and Revenues
missing_mask = (df_films_aux['tmdb_budget'] == 0) | (df_films_aux['tmdb_revenue'] == 0)

cols_to_view = ['film_id', 'film_title', 'tmdb_budget', 'tmdb_revenue']

df_missing = df_films_aux.loc[missing_mask, cols_to_view]
print(df_missing.to_string())

     film_id                            film_title  tmdb_budget  tmdb_revenue
1     193402                      She Loves Me Not            0             0
2     193501                  Gold Diggers of 1935            0             0
4     193503                               Roberta            0             0
6     193602                         Born to Dance            0             0
7     193603                   Pennies from Heaven            0             0
8     193604                      Sing, Baby, Sing            0             0
9     193605                                  Suzy            0             0
10    193701                       Waikiki Wedding            0             0
12    193703                        Vogues of 1938            0             0
13    193704                    Artists and Models            0             0
14    193801             The Big Broadcast of 1938      1000000             0
16    193803                          Going Places            0 

## TRANSFORMACIONES

### Columnas de Década

In [11]:
df_songs_aux['decade'] = (df_songs_aux['year'] // 10) * 10

### Imputaciones de Datos

Incorporo algún dato de streams de canción que se pasó en la extracción inicial (para no repetir todo el proceso que puede tardar 90 minutos aprox.)

In [12]:
# Fill Missing Stream data
df_songs_aux.loc[df_songs_aux['song_id'] == 199505, ['other_artists', 'streams_others']] = ['Sheléa', 808000]

# Display Confirmation
display(df_songs_aux.loc[df_songs_aux['song_id'] == 199505, ['song_title', 'other_artists', 'streams_others']])

,song_title,other_artists,streams_others
342,Moonlight,Sheléa,808000.0


También incorporo nuevos datos de presupuestos y taquilla de Películas que no estaban disponibles en la API de TMDB o OMDB pero pude captar manualmente en la web de IMDB.

In [13]:
# Fill Missing Budgets and Revenues (from IMDB)
df_films_aux.loc[df_films_aux['film_id'] == 193806, ['tmdb_budget', 'tmdb_revenue']] = [1500000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 194006, ['tmdb_budget', 'tmdb_revenue']] = [840000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 194007, ['tmdb_budget', 'tmdb_revenue']] = [353000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 194101, ['tmdb_budget', 'tmdb_revenue']] = [863000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 194104, ['tmdb_budget', 'tmdb_revenue']] = [940000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 194106, ['tmdb_budget', 'tmdb_revenue']] = [245000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 194201, ['tmdb_budget', 'tmdb_revenue']] = [3200000, 3750000]
df_films_aux.loc[df_films_aux['film_id'] == 194202, ['tmdb_budget', 'tmdb_revenue']] = [0, 673000]
df_films_aux.loc[df_films_aux['film_id'] == 194301, ['tmdb_budget', 'tmdb_revenue']] = [0, 3400000]
df_films_aux.loc[df_films_aux['film_id'] == 194302, ['tmdb_budget', 'tmdb_revenue']] = [0, 602000]
df_films_aux.loc[df_films_aux['film_id'] == 194401, ['tmdb_budget', 'tmdb_revenue']] = [0, 2200]
df_films_aux.loc[df_films_aux['film_id'] == 194504, ['tmdb_budget', 'tmdb_revenue']] = [2510000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 194601, ['tmdb_budget', 'tmdb_revenue']] = [2525000, 1400]
df_films_aux.loc[df_films_aux['film_id'] == 194602, ['tmdb_budget', 'tmdb_revenue']] = [3000000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 194803, ['tmdb_budget', 'tmdb_revenue']] = [1307000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 195101, ['tmdb_budget', 'tmdb_revenue']] = [2120000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 195102, ['tmdb_budget', 'tmdb_revenue']] = [885000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 195103, ['tmdb_budget', 'tmdb_revenue']] = [1528000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 195203, ['tmdb_budget', 'tmdb_revenue']] = [4000000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 195301, ['tmdb_budget', 'tmdb_revenue']] = [0, 9200]
df_films_aux.loc[df_films_aux['film_id'] == 195302, ['tmdb_budget', 'tmdb_revenue']] = [1864000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 195303, ['tmdb_budget', 'tmdb_revenue']] = [1440000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 195401, ['tmdb_budget', 'tmdb_revenue']] = [0, 6800]
df_films_aux.loc[df_films_aux['film_id'] == 195404, ['tmdb_budget', 'tmdb_revenue']] = [1470000, 4300]
df_films_aux.loc[df_films_aux['film_id'] == 195405, ['tmdb_budget', 'tmdb_revenue']] = [0, 2990000]
df_films_aux.loc[df_films_aux['film_id'] == 195501, ['tmdb_budget', 'tmdb_revenue']] = [1780000, 29300]
df_films_aux.loc[df_films_aux['film_id'] == 195505, ['tmdb_budget', 'tmdb_revenue']] = [2750000, 200]
df_films_aux.loc[df_films_aux['film_id'] == 195604, ['tmdb_budget', 'tmdb_revenue']] = [785000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 195803, ['tmdb_budget', 'tmdb_revenue']] = [3500000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 195805, ['tmdb_budget', 'tmdb_revenue']] = [3150000, 30800]
df_films_aux.loc[df_films_aux['film_id'] == 195901, ['tmdb_budget', 'tmdb_revenue']] = [1890000, 2900]
df_films_aux.loc[df_films_aux['film_id'] == 195902, ['tmdb_budget', 'tmdb_revenue']] = [1350000, 9000]
df_films_aux.loc[df_films_aux['film_id'] == 195903, ['tmdb_budget', 'tmdb_revenue']] = [2500000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 196002, ['tmdb_budget', 'tmdb_revenue']] = [0, 5450000]
df_films_aux.loc[df_films_aux['film_id'] == 196005, ['tmdb_budget', 'tmdb_revenue']] = [0, 9600000]
df_films_aux.loc[df_films_aux['film_id'] == 196103, ['tmdb_budget', 'tmdb_revenue']] = [1990000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 196201, ['tmdb_budget', 'tmdb_revenue']] = [0, 2000]
df_films_aux.loc[df_films_aux['film_id'] == 196202, ['tmdb_budget', 'tmdb_revenue']] = [3000000, 3000000]
df_films_aux.loc[df_films_aux['film_id'] == 196312, ['tmdb_budget', 'tmdb_revenue']] = [3000000, 10878107]
df_films_aux.loc[df_films_aux['film_id'] == 196404, ['tmdb_budget', 'tmdb_revenue']] = [0, 9800000]
df_films_aux.loc[df_films_aux['film_id'] == 196501, ['tmdb_budget', 'tmdb_revenue']] = [5300000, 6161000]
df_films_aux.loc[df_films_aux['film_id'] == 196602, ['tmdb_budget', 'tmdb_revenue']] = [800000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 196604, ['tmdb_budget', 'tmdb_revenue']] = [400000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 196703, ['tmdb_budget', 'tmdb_revenue']] = [6000000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 196805, ['tmdb_budget', 'tmdb_revenue']] = [2590000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 196902, ['tmdb_budget', 'tmdb_revenue']] = [0, 275000]
df_films_aux.loc[df_films_aux['film_id'] == 196903, ['tmdb_budget', 'tmdb_revenue']] = [2000000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 196905, ['tmdb_budget', 'tmdb_revenue']] = [2750000, 14500]
df_films_aux.loc[df_films_aux['film_id'] == 197001, ['tmdb_budget', 'tmdb_revenue']] = [0, 5484000]
df_films_aux.loc[df_films_aux['film_id'] == 197103, ['tmdb_budget', 'tmdb_revenue']] = [0, 212000]
df_films_aux.loc[df_films_aux['film_id'] == 197104, ['tmdb_budget', 'tmdb_revenue']] = [3660000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 197202, ['tmdb_budget', 'tmdb_revenue']] = [0, 770000]
df_films_aux.loc[df_films_aux['film_id'] == 197203, ['tmdb_budget', 'tmdb_revenue']] = [0, 16500000]
df_films_aux.loc[df_films_aux['film_id'] == 197305, ['tmdb_budget', 'tmdb_revenue']] = [2465000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 197403, ['tmdb_budget', 'tmdb_revenue']] = [500000, 39550000]
df_films_aux.loc[df_films_aux['film_id'] == 197504, ['tmdb_budget', 'tmdb_revenue']] = [0, 5000000]
df_films_aux.loc[df_films_aux['film_id'] == 197703, ['tmdb_budget', 'tmdb_revenue']] = [10000000, 39600000]
df_films_aux.loc[df_films_aux['film_id'] == 197801, ['tmdb_budget', 'tmdb_revenue']] = [2200000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 197805, ['tmdb_budget', 'tmdb_revenue']] = [0, 20700000]
df_films_aux.loc[df_films_aux['film_id'] == 197903, ['tmdb_budget', 'tmdb_revenue']] = [0, 11600000]
df_films_aux.loc[df_films_aux['film_id'] == 198003, ['tmdb_budget', 'tmdb_revenue']] = [11000000, 17800000]
df_films_aux.loc[df_films_aux['film_id'] == 198005, ['tmdb_budget', 'tmdb_revenue']] = [10100000, 14285000]
df_films_aux.loc[df_films_aux['film_id'] == 198102, ['tmdb_budget', 'tmdb_revenue']] = [9700000, 32492674]
df_films_aux.loc[df_films_aux['film_id'] == 198104, ['tmdb_budget', 'tmdb_revenue']] = [0, 14920000]
df_films_aux.loc[df_films_aux['film_id'] == 198402, ['tmdb_budget', 'tmdb_revenue']] = [13000000, 21690000]
df_films_aux.loc[df_films_aux['film_id'] == 198501, ['tmdb_budget', 'tmdb_revenue']] = [20000000, 42160849]
df_films_aux.loc[df_films_aux['film_id'] == 198604, ['tmdb_budget', 'tmdb_revenue']] = [10000000, 4080000]
df_films_aux.loc[df_films_aux['film_id'] == 198803, ['tmdb_budget', 'tmdb_revenue']] = [0, 540000]
df_films_aux.loc[df_films_aux['film_id'] == 198905, ['tmdb_budget', 'tmdb_revenue']] = [0, 6000000]
df_films_aux.loc[df_films_aux['film_id'] == 199112, ['tmdb_budget', 'tmdb_revenue']] = [17000000, 7240000]
df_films_aux.loc[df_films_aux['film_id'] == 199204, ['tmdb_budget', 'tmdb_revenue']] = [15500000, 6742168]
df_films_aux.loc[df_films_aux['film_id'] == 199713, ['tmdb_budget', 'tmdb_revenue']] = [0, 26700]
df_films_aux.loc[df_films_aux['film_id'] == 200205, ['tmdb_budget', 'tmdb_revenue']] = [25000000, 60700000]
df_films_aux.loc[df_films_aux['film_id'] == 201313, ['tmdb_budget', 'tmdb_revenue']] = [0, 300000]
df_films_aux.loc[df_films_aux['film_id'] == 201505, ['tmdb_budget', 'tmdb_revenue']] = [1500000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 201604, ['tmdb_budget', 'tmdb_revenue']] = [0, 8250]
df_films_aux.loc[df_films_aux['film_id'] == 201703, ['tmdb_budget', 'tmdb_revenue']] = [10000000, 86000]
df_films_aux.loc[df_films_aux['film_id'] == 201903, ['tmdb_budget', 'tmdb_revenue']] = [17000000, 43350000]
df_films_aux.loc[df_films_aux['film_id'] == 202004, ['tmdb_budget', 'tmdb_revenue']] = [17000000, 0]
df_films_aux.loc[df_films_aux['film_id'] == 202212, ['tmdb_budget', 'tmdb_revenue']] = [7000000, 12800]
df_films_aux.loc[df_films_aux['film_id'] == 202404, ['tmdb_budget', 'tmdb_revenue']] = [2000000, 3110476]
# df_films_aux.loc[df_films_aux['film_id'] == 00, ['tmdb_budget', 'tmdb_revenue']] = [0, 0]

# Display imputations
print(df_films_aux[['film_id', 'film_title', 'tmdb_budget', 'tmdb_revenue']].to_string())

     film_id                                         film_title  tmdb_budget  tmdb_revenue
0     193401                                   The Gay Divorcee       520000       1800000
1     193402                                   She Loves Me Not            0             0
2     193501                               Gold Diggers of 1935            0             0
3     193502                                            Top Hat       609000       3202000
4     193503                                            Roberta            0             0
5     193601                                         Swing Time       886000       2600000
6     193602                                      Born to Dance            0             0
7     193603                                Pennies from Heaven            0             0
8     193604                                   Sing, Baby, Sing            0             0
9     193605                                               Suzy            0             0

### Eliminación de datos inservibles

Decido eliminar Canciones donde no dispongo de ningún tipo de dato de streams o reproducciones, puesto que la canción no fue localizada ni a través de las API ni en búsqueda manual en Spotify.

In [14]:
# Display Missing Streams and Playcounts
missing_mask = (df_songs_aux['streams_original'].isna() & df_songs_aux['streams_others'].isna() 
                & df_songs_aux['lastfm_track_playcount'].isna() & df_songs_aux['lastfm_track_listeners'].isna())

cols_to_view = [
    'song_id', 'song_title',
    'streams_original', 'streams_others',
    'lastfm_track_playcount','lastfm_track_listeners'
]

# Apply the filter and view the results
df_missing = df_songs_aux.loc[missing_mask, cols_to_view]
display(df_missing)

,song_id,song_title,streams_original,streams_others,lastfm_track_playcount,lastfm_track_listeners
257,198304,Over You,NaN,NaN,NaN,NaN
315,199005,I'm Checkin' Out,NaN,NaN,NaN,NaN
337,199405,Look What Love Has Done,NaN,NaN,NaN,NaN
396,200405,In The Deep,NaN,NaN,NaN,NaN


In [15]:
# Delete the invalid songs
delete_mask = ((df_songs_aux['song_id'] == 198304) | (df_songs_aux['song_id'] == 199005)
               | (df_songs_aux['song_id'] == 199405) | (df_songs_aux['song_id'] == 199405))

df_songs_aux.drop(df_songs_aux[delete_mask].index, inplace=True)
df_songs_aux.reset_index(drop=True, inplace=True)

display(df_songs_aux[['song_id', 'song_title', 
                      'streams_original', 'streams_others',
                      'lastfm_track_playcount','lastfm_track_listeners']].info())

<class 'pandas.DataFrame'>
RangeIndex: 497 entries, 0 to 496
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   song_id                 497 non-null    int64  
 1   song_title              497 non-null    str    
 2   streams_original        469 non-null    float64
 3   streams_others          105 non-null    float64
 4   lastfm_track_playcount  487 non-null    float64
 5   lastfm_track_listeners  487 non-null    float64
dtypes: float64(4), int64(1), str(1)
memory usage: 32.0 KB


None

### Columnas Booleanas

Transformamos a valores booleanos algunas columnas que habían sido extraídas como texto (Y / N)

In [16]:
df_songs_aux = df_songs_aux.assign(
    oscar_song_nominee = lambda x: x['oscar_song_nominee'] == 'Y',
    oscar_song_win = lambda x: x['oscar_song_win'] == 'Y',
    grammy_song_nominee = lambda x: x['grammy_song_nominee'] == 'Y',
    grammy_song_win = lambda x: x['grammy_song_win'] == 'Y',
    grammy_record_nominee = lambda x: x['grammy_record_nominee'] == 'Y',
    grammy_record_win = lambda x: x['grammy_record_win'] == 'Y'
)

display(df_songs_aux[['song_title', 'oscar_song_nominee', 'oscar_song_win', 'grammy_song_nominee', 'grammy_song_win']].head(10))

,song_title,oscar_song_nominee,oscar_song_win,grammy_song_nominee,grammy_song_win
0,The Continental,True,True,False,False
1,Love In Bloom,True,False,False,False
2,Lullaby Of Broadway,True,True,False,False
3,Cheek To Cheek,True,False,False,False
4,Lovely To Look At,True,False,False,False
5,The Way You Look Tonight,True,True,False,False
6,I've Got You Under My Skin,True,False,False,False
7,Pennies From Heaven,True,False,False,False
8,When Did You Leave Heaven,True,False,False,False
9,Did I Remember,True,False,False,False


### Conversión tipos de datos

In [17]:
metric_cols = [
    'streams_original', 
    'streams_others', 
    'lastfm_track_playcount', 
    'lastfm_track_listeners'
]

# Apply to_numeric and cast to Int64
for col in metric_cols:
    df_songs_aux[col] = pd.to_numeric(df_songs_aux[col], errors='coerce').astype('Int64')

# print(df_songs_aux[metric_cols].head().to_string())
print("\nData Types:")
print(df_songs_aux[metric_cols].dtypes)


Data Types:
streams_original          Int64
streams_others            Int64
lastfm_track_playcount    Int64
lastfm_track_listeners    Int64
dtype: object


In [18]:
metric_cols = [
    'streams_original', 
    'streams_others', 
    'lastfm_track_playcount', 
    'lastfm_track_listeners'
]

# Cleanup Zeroes
# Replace exactly 0 with pd.NA across all specified columns
df_songs_aux[metric_cols] = df_songs_aux[metric_cols].replace(0, pd.NA)

# Check how many zeros were successfully converted to missing values
print("Missing values after converting zeros:")
print(df_songs_aux[metric_cols].isna().sum())

Missing values after converting zeros:
streams_original           28
streams_others            392
lastfm_track_playcount     12
lastfm_track_listeners     12
dtype: int64


### Imputación columna de Streams

In [19]:
# Imputate the maximum between the two columns for each row
df_songs_aux['streams_obs_max'] = (
    df_songs_aux[['streams_original', 'streams_others']]
    .max(axis=1)
    .astype('Int64') # Maintain our nullable integer type
)

print(df_songs_aux[['streams_original', 'streams_others', 'streams_obs_max']].head(10))
# print(df_songs_aux[['streams_obs_max']].describe())
# print(df_songs_aux[['streams_obs_max']].info())

   streams_original  streams_others  streams_obs_max
0             65000            <NA>            65000
1             15800          171500           171500
2             37200         7020000          7020000
3          28200000       210100000        210100000
4            218000            <NA>           218000
5           7000000       269900000        269900000
6            103400       268000000        268000000
7           1000000        73700000         73700000
8              <NA>          348000           348000
9           1200000        34500000         34500000


### Normalización Streams

In [20]:
# Percentiles and Logarithmic Normalization of Streams

# 1. Percentile to 0-100 Scale
df_songs_aux['streams_percentile'] = (
    df_songs_aux['streams_obs_max'].rank(pct=True) * 100
).round(2)

# 2. Logarithmic to 0-100 Scale (Min-Max Scaling)
# First, get the raw log values
log_streams = np.log10(df_songs_aux['streams_obs_max'].astype(float))

# Calculate the minimum and maximum of those log values
log_min = log_streams.min()
log_max = log_streams.max()

# Apply Min-Max scaling: (value - min) / (max - min) * 100
df_songs_aux['streams_normalized'] = (
    ((log_streams - log_min) / (log_max - log_min)) * 100
).round(2)

print(df_songs_aux[['streams_obs_max', 'streams_percentile', 'streams_normalized']].head(10))

   streams_obs_max  streams_percentile  streams_normalized
0            65000                8.69               26.93
1           171500               11.72               33.32
2          7020000               43.23               57.74
3        210100000               79.19               80.11
4           218000               12.93               34.89
5        269900000               81.62               81.75
6        268000000               81.41               81.71
7         73700000               69.49               73.21
8           348000               15.76               37.97
9         34500000               59.19               68.22


In [21]:
lastfm_cols = ['lastfm_track_playcount', 'lastfm_track_listeners']

for col in lastfm_cols:
    # 1. Percentile to 0-100 Scale
    df_songs_aux[f'{col}_percentile'] = (
        df_songs_aux[col].rank(pct=True) * 100
    ).round(2)
    
    # 2. Logarithmic to 0-100 Scale (Min-Max)
    # Cast to float for np.log10
    log_vals = np.log10(df_songs_aux[col].astype(float))
    log_min = log_vals.min()
    log_max = log_vals.max()
    
    df_songs_aux[f'{col}_normalized'] = (
        ((log_vals - log_min) / (log_max - log_min)) * 100
    ).round(2)

print(df_songs_aux[[
    'lastfm_track_playcount', 
    'lastfm_track_playcount_percentile', 
    'lastfm_track_playcount_normalized',
    'lastfm_track_listeners_percentile',
    'lastfm_track_listeners_normalized'
]].head(10).to_string())

   lastfm_track_playcount  lastfm_track_playcount_percentile  lastfm_track_playcount_normalized  lastfm_track_listeners_percentile  lastfm_track_listeners_normalized
0                   19965                              27.01                              57.80                              33.81                              63.06
1                    6003                              18.35                              50.79                              20.21                              53.57
2                   47022                              37.11                              62.80                              44.12                              68.13
3                 1765062                              85.77                              83.97                              89.48                              89.58
4                    3571                              15.67                              47.75                              17.53                              49.66
5   

### Normalización Datos de AM y CM

In [22]:
# Chart Masters and Acclaimed Music metric normalization

# ---------------------------------------------------------
# 1. Normalizing Equivalent Album Sales (EAS)
# ---------------------------------------------------------
col_eas = 'cm_song_eas'

# Fill missing sales with 0
eas_filled = df_songs_aux[col_eas].fillna(0).astype(float)

# Percentile
df_songs_aux[f'{col_eas}_percentile'] = (
    eas_filled.rank(pct=True) * 100
).round(2)

# Apply Log Shift (+1) because we now have true zeros
log_vals_eas = np.log10(eas_filled + 1)
log_min_eas = log_vals_eas.min()
log_max_eas = log_vals_eas.max()

df_songs_aux[f'{col_eas}_normalized'] = (
    ((log_vals_eas - log_min_eas) / (log_max_eas - log_min_eas)) * 100
).round(2)

# ---------------------------------------------------------
# 2. Normalizing the Ranking Columns (AM & CM)
# ---------------------------------------------------------
rank_cols = ['am_ranking', 'cm_ranking']

for col in rank_cols:
    # Find the worst (highest number) rank on the chart
    worst_rank = df_songs_aux[col].max()
    
    # Fill unranked songs with worst_rank + 1
    rank_filled = df_songs_aux[col].fillna(worst_rank + 1).astype(float)
    
    # A. Inverted Percentile
    df_songs_aux[f'{col}_percentile'] = (
        rank_filled.rank(pct=True, method='min', ascending=False) * 100
    ).round(2)
    
    # B. Inverted Logarithmic Min-Max
    log_vals = np.log10(rank_filled)
    log_min = log_vals.min()
    log_max = log_vals.max()
    
    df_songs_aux[f'{col}_normalized'] = (
        100 - (((log_vals - log_min) / (log_max - log_min)) * 100)
    ).round(2)

# Verify the inversion
# A song with cm_ranking of 1 should have a normalized score of 100.
cols_to_check = [
    'am_ranking', 'am_ranking_percentile', 'am_ranking_normalized',
    'cm_ranking', 'cm_ranking_percentile', 'cm_ranking_normalized',
    'cm_song_eas', 'cm_song_eas_normalized'
]
print(df_songs_aux.sort_values('cm_ranking')[cols_to_check].head(15).to_string())

     am_ranking  am_ranking_percentile  am_ranking_normalized  cm_ranking  cm_ranking_percentile  cm_ranking_normalized  cm_song_eas  cm_song_eas_normalized
347        3055                  93.16                  27.65           1                 100.00                 100.00     64042000                  100.00
216         138                 100.00                 100.00           9                  99.80                  76.10     34307000                   96.53
317        4759                  90.54                  17.30          39                  99.60                  60.16     21807000                   94.01
161         786                  97.99                  59.36          53                  99.40                  56.82     19677000                   93.43
39          336                  98.59                  79.21          55                  99.20                  56.42     19602000                   93.41
105         160                  99.60                  96

### Normalización de Song Awards

In [23]:
# Create a column with the major Song awards
df_songs_aux['song_major_awards'] = (df_songs_aux['oscar_song_win'].astype('Int64') 
                                     + df_songs_aux['grammy_song_win'].astype('Int64')
                                     + df_songs_aux['grammy_record_win'].astype('Int64'))

df_songs_aux['song_major_awards'] = df_songs_aux['song_major_awards'].astype('Int64')

# Assign an Awards combined value percent

# 1. Calculate the weighted score (max 10)
df_songs_aux['awards_combined'] = (
    (df_songs_aux['oscar_song_nominee'] * 3) +
    (df_songs_aux['oscar_song_win'] * 3) +
    (df_songs_aux['grammy_song_nominee'] * 1) +
    (df_songs_aux['grammy_song_win'] * 1) +
    (df_songs_aux['grammy_record_nominee'] * 1) +
    (df_songs_aux['grammy_record_win'] * 1)
)

# 2. Scale it to 0-100 to match your other percentile/normalized metrics
df_songs_aux['awards_combined_percent'] = df_songs_aux['awards_combined'] * 10

# Cast the new percentage column to a nullable float
df_songs_aux['awards_combined_percent'] = df_songs_aux['awards_combined_percent'].astype('Float64')

print(df_songs_aux[['song_title', 'song_major_awards', 'awards_combined_percent']].head().to_string())

            song_title  song_major_awards  awards_combined_percent
0      The Continental                  1                     60.0
1        Love In Bloom                  0                     30.0
2  Lullaby Of Broadway                  1                     60.0
3       Cheek To Cheek                  0                     30.0
4    Lovely To Look At                  0                     30.0


### Normalización datos de Recordings

In [24]:
# Calculate percentiles (0.0 to 1.0) and multiply by 100 for a 0-100 scale
df_songs_aux['mb_original_recordings_pct'] = (
    df_songs_aux['mb_original_recordings_count'].rank(pct=True) * 100
)

df_songs_aux['mb_cover_recordings_pct'] = (
    df_songs_aux['mb_cover_recordings_count'].rank(pct=True) * 100
)

# Preview the new metrics alongside the old ones to verify
print(df_songs_aux[[
    'mb_original_recordings_count', 'mb_original_recordings_pct', 
    'mb_cover_recordings_count', 'mb_cover_recordings_pct'
]].describe().to_string())

       mb_original_recordings_count  mb_original_recordings_pct  mb_cover_recordings_count  mb_cover_recordings_pct
count                    487.000000                  487.000000                 487.000000               487.000000
mean                      20.406571                   50.102669                  54.993840                50.102669
std                       44.190653                   28.785153                 122.023298                28.819429
min                        0.000000                    7.494867                   0.000000                 8.316222
25%                        1.000000                   22.587269                   1.000000                20.944559
50%                        5.000000                   49.383984                  14.000000                50.410678
75%                       21.000000                   74.845996                  54.000000                75.051335
max                      378.000000                  100.000000         

In [25]:
# Recordings Factor Normalization

# Step 1: Calculate the weighted raw score first
df_songs_aux['weighted_recordings_raw'] = (
    df_songs_aux['mb_original_recordings_count'].fillna(0) + 
    (df_songs_aux['mb_cover_recordings_count'].fillna(0) * 2)
)

# Step 2: Identify the true extraction failures (where the sum of both columns is 0)
zero_totals_mask = df_songs_aux[['mb_original_recordings_count', 'mb_cover_recordings_count']].sum(axis=1) == 0

# Step 3: Overwrite the weighted score with NaN only for those failed extractions
df_songs_aux.loc[zero_totals_mask, 'weighted_recordings_raw'] = np.nan

# Step 4: Calculate the final percentile metric
df_songs_aux['recordings_metric_pct'] = (
    df_songs_aux['weighted_recordings_raw'].rank(pct=True) * 100
)

df_songs_aux['mb_original_recordings_pct'] = df_songs_aux['mb_original_recordings_pct'].round(2)
df_songs_aux['mb_cover_recordings_pct'] = df_songs_aux['mb_cover_recordings_pct'].round(2)
df_songs_aux['recordings_metric_pct'] = df_songs_aux['recordings_metric_pct'].round(2)

# Check how many valid scores survived the filter
print("Valid scores remaining:", df_songs_aux['recordings_metric_pct'].notna().sum())

Valid scores remaining: 460


### Datos de Film Awards

In [26]:
print(df_films_aux[['film_title', 'omdb_awards']].head())

             film_title                                        omdb_awards
0      The Gay Divorcee          Won 1 Oscar. 4 wins & 5 nominations total
1      She Loves Me Not  Nominated for 1 Oscar. 1 win & 1 nomination total
2  Gold Diggers of 1935            Won 1 Oscar. 1 win & 1 nomination total
3               Top Hat  Nominated for 4 Oscars. 7 wins & 6 nominations...
4               Roberta         Nominated for 1 Oscar. 2 nominations total


In [27]:
# Awards Calculations

# Columns tidy up
df_films_aux['oscar_wins'] = df_films_aux['oscar_wins'].fillna(0).astype(int)
df_films_aux['oscar_nominations'] = df_films_aux['oscar_nominations'].fillna(0).astype(int)

# ==========================================
# REGEX EXTRACTION
# ==========================================
# r'...'   -> Tells Python this is a "raw string" regex pattern
# (\d+)    -> The capture group. \d means "digits", + means "one or more". It grabs the number!
# \s*      -> Accounts for any spaces (or lack of spaces) between the number and the word
# win      -> The exact text to look for right after the number. 

df_films_aux['total_wins'] = df_films_aux['omdb_awards'].str.extract(r'(\d+)\s*win', expand=False)
df_films_aux['total_nominations'] = df_films_aux['omdb_awards'].str.extract(r'(\d+)\s*nomination', expand=False)

# Fill NaN with 0
df_films_aux['total_wins'] = df_films_aux['total_wins'].fillna(0)
df_films_aux['total_nominations'] = df_films_aux['total_nominations'].fillna(0)

# Convert to standard integers so you can do math on them later
df_films_aux['total_wins'] = df_films_aux['total_wins'].astype(int)
df_films_aux['total_nominations'] = df_films_aux['total_nominations'].astype(int)

# ==========================================
# LOGICAL RECALCULATION (The "True" Nominations)
# ==========================================
# Since OMDB treats "wins" and "nominations" as mutually exclusive,
# we add the wins to the nominations to get the chronological total.
df_films_aux['total_nominations'] = df_films_aux['total_nominations'] + df_films_aux['total_wins']

# Drop raw column
# df_films_aux = df_films_aux.drop(columns=['omdb_awards'])

# Display resulting dataframe
print(df_films_aux[['film_title', 'oscar_wins', 'oscar_nominations', 'total_wins', 'total_nominations']].head().to_string())

             film_title  oscar_wins  oscar_nominations  total_wins  total_nominations
0      The Gay Divorcee           1                  5           4                  9
1      She Loves Me Not           0                  1           1                  2
2  Gold Diggers of 1935           1                  2           1                  2
3               Top Hat           0                  4           7                 13
4               Roberta           0                  1           0                  2


In [28]:
# Song Awards Calculation
nom_columns = ['oscar_song_nominee', 'grammy_song_nominee', 'grammy_record_nominee']
win_columns = ['oscar_song_win', 'grammy_song_win', 'grammy_record_win']

df_songs_aux['song_nominations'] = df_songs_aux[nom_columns].sum(axis=1)
df_songs_aux['song_wins'] = df_songs_aux[win_columns].sum(axis=1)

display(df_songs_aux[['song_title', 'song_nominations', 'song_wins']].head())

,song_title,song_nominations,song_wins
0,The Continental,1,1
1,Love In Bloom,1,0
2,Lullaby Of Broadway,1,1
3,Cheek To Cheek,1,0
4,Lovely To Look At,1,0


### Ajustar Datos Monetarios

In [29]:
df_films_aux['tmdb_budget'] = pd.to_numeric(df_films_aux['tmdb_budget'], errors='coerce')
df_films_aux['tmdb_revenue'] = pd.to_numeric(df_films_aux['tmdb_revenue'], errors='coerce')

df_films_aux['tmdb_budget'] = df_films_aux['tmdb_budget'].replace(0, np.nan)
df_films_aux['tmdb_revenue'] = df_films_aux['tmdb_revenue'].replace(0, np.nan)

print("Calculating historical multipliers...")

# Step A: Find the unique years in your dataset
unique_years = df_films_aux['year'].dropna().unique()

# Step B: Create a dictionary of multipliers ($1 in that year -> Current Dollars)
cpi_multipliers = {}

for year in unique_years:
    try:
        # Calculate what $1 from this release year is worth today
        cpi_multipliers[year] = cpi.inflate(1, int(year))
    except Exception as e:
        print(f"Warning: Could not get CPI for {year}. Defaulting to 1.0 multiplier.")
        cpi_multipliers[year] = 1.0

# Step C: Map the dictionary back to the DataFrame to create a new multiplier column
df_films_aux['cpi_multiplier'] = df_films_aux['year'].map(cpi_multipliers)

# Multiply the entire column
df_films_aux['budget_adjusted'] = df_films_aux['tmdb_budget'] * df_films_aux['cpi_multiplier']
df_films_aux['revenue_adjusted'] = df_films_aux['tmdb_revenue'] * df_films_aux['cpi_multiplier']

# Round the results and convert to Nullable Integers
df_films_aux['budget_adjusted'] = df_films_aux['budget_adjusted'].round().astype('Int64')
df_films_aux['revenue_adjusted'] = df_films_aux['revenue_adjusted'].round().astype('Int64')

# Drop the auxiliar multiplier column
# df_films_aux = df_films_aux.drop(columns=['cpi_multiplier'])

# Show the results
print(df_films_aux[['film_title', 'year', 'tmdb_budget', 'budget_adjusted']])

Calculating historical multipliers...
                   film_title  year  tmdb_budget  budget_adjusted
0            The Gay Divorcee  1934     520000.0         12493310
1            She Loves Me Not  1934          NaN             <NA>
2        Gold Diggers of 1935  1935          NaN             <NA>
3                     Top Hat  1935     609000.0         14311189
4                     Roberta  1935          NaN             <NA>
..                        ...   ...          ...              ...
462        KPop Demon Hunters  2025  100000000.0        100000000
463                   Sinners  2025  100000000.0        100000000
464              Train Dreams  2025   10000000.0         10000000
465               Viva Verdi!  2025          NaN             <NA>
466  Diane Warren: Relentless  2025          NaN             <NA>

[467 rows x 4 columns]


In [30]:
# ==========================================
# DATA COVERAGE REPORT
# ==========================================
# Let's see how much data we have per decade to understand the gaps
df_films_aux['decade'] = (df_films_aux['year'] // 10) * 10

# Group by decade and calculate the percentage of non-null budgets
coverage = df_films_aux.groupby('decade')['tmdb_budget'].apply(lambda x: x.count() / len(x) * 100)

print("=== BUDGET DATA COVERAGE BY DECADE (%) ===")
print(coverage.round(1))

# TIP: How to handle the gaps in your Final Analysis
# Whenever you do an analysis, define a "Data Integrity" subset if needed:
# df_complete_financials = df_films_aux.dropna(subset=['budget_adjusted', 'revenue_adjusted'])

print("\nAnalysis Tip: There are " + str(len(df_films_aux.dropna(subset=['budget_adjusted']))) + " movies with full financial data.")

=== BUDGET DATA COVERAGE BY DECADE (%) ===
decade
1930    39.1
1940    44.0
1950    62.8
1960    70.7
1970    73.6
1980    92.4
1990    96.3
2000    95.3
2010    85.7
2020    71.4
Name: tmdb_budget, dtype: float64

Analysis Tip: There are 354 movies with full financial data.


### Cálculos de Factores Financieros

In [31]:
df_films_aux['budget_math'] = df_films_aux['budget_adjusted'].astype(float)
df_films_aux['revenue_math'] = df_films_aux['revenue_adjusted'].astype(float)

# Raw ROI Percentage
# Formula: (Revenue - Budget) / Budget
df_films_aux['roi_percent'] = ((df_films_aux['revenue_math'] - df_films_aux['budget_math']) / df_films_aux['budget_math']) * 100

# Commercial Success Factor (The "2.5x Rule")
# Formula: Revenue / (Budget * 2.5)
# This tells us if they exceeded the "break-even" threshold
df_films_aux['commercial_success_factor'] = df_films_aux['revenue_math'] / (df_films_aux['budget_math'] * 2.5)

# Cleanup and Format
# Round for readability (ROI as percentage, Factor as a decimal)
df_films_aux['roi_percent'] = df_films_aux['roi_percent'].round(1)
df_films_aux['commercial_success_factor'] = df_films_aux['commercial_success_factor'].round(2)

# Remove the temporary math columns
df_films_aux = df_films_aux.drop(columns=['budget_math', 'revenue_math'])

print("=== ROI AND COMMERCIAL SUCCESS METRICS ===")
print(df_films_aux[['film_title', 'budget_adjusted', 'revenue_adjusted', 'roi_percent', 'commercial_success_factor']].head(10).to_string())

=== ROI AND COMMERCIAL SUCCESS METRICS ===
             film_title  budget_adjusted  revenue_adjusted  roi_percent  commercial_success_factor
0      The Gay Divorcee         12493310          43246075        246.2                       1.38
1      She Loves Me Not             <NA>              <NA>          NaN                        NaN
2  Gold Diggers of 1935             <NA>              <NA>          NaN                        NaN
3               Top Hat         14311189          75245364        425.8                       2.10
4               Roberta             <NA>              <NA>          NaN                        NaN
5            Swing Time         20520971          60219554        193.5                       1.17
6         Born to Dance             <NA>              <NA>          NaN                        NaN
7   Pennies from Heaven             <NA>              <NA>          NaN                        NaN
8      Sing, Baby, Sing             <NA>              <NA>        

### Normalización de métricas Financieras

In [32]:
# Financial Metrics Normalization
financial_cols = [
    'budget_adjusted', 'revenue_adjusted', 
    'roi_percent', 'commercial_success_factor'
]

for col in financial_cols:
    # 1. Percentile Normalization (This works perfectly with negatives/zeros!)
    df_films_aux[f'{col}_percentile'] = (
        df_films_aux[col].rank(pct=True) * 100
    ).round(2)
    
    # 2. Logarithmic Min-Max Normalization
    # Check if the column has any values <= 0
    min_val = df_films_aux[col].min()
    
    if min_val <= 0:
        # LOG SHIFT: Shift the data so the minimum value is exactly 1
        shift_amount = abs(min_val) + 1
        shifted_series = df_films_aux[col].dropna().astype(float) + shift_amount
        log_vals = np.log10(shifted_series)
    else:
        # Standard approach for purely positive columns
        log_vals = np.log10(df_films_aux[col].dropna().astype(float))
        
    log_min = log_vals.min()
    log_max = log_vals.max()
    
    df_films_aux[f'{col}_normalized'] = (
        ((log_vals - log_min) / (log_max - log_min)) * 100
    ).round(2)

# Verify the columns with the tricky zero/negative minimums
cols_to_check = [
    'roi_percent', 'roi_percent_percentile', 'roi_percent_normalized',
    'commercial_success_factor', 'commercial_success_factor_percentile', 'commercial_success_factor_normalized'
]
print(df_films_aux.sort_values('roi_percent')[cols_to_check].head(20).to_string())

     roi_percent  roi_percent_percentile  roi_percent_normalized  commercial_success_factor  commercial_success_factor_percentile  commercial_success_factor_normalized
95        -100.0                    0.31                    0.00                       0.00                                  1.71                                  0.00
60         -99.9                    0.62                    0.92                       0.00                                  1.71                                  0.00
453        -99.8                    1.09                    1.76                       0.00                                  1.71                                  0.00
111        -99.8                    1.09                    1.76                       0.00                                  1.71                                  0.00
89         -99.7                    1.71                    2.54                       0.00                                  1.71                               

### Normalización Movie Awards

In [39]:
# Calculate the Movie Awards Normalized score

# 1. Calculate the raw weighted prestige score and cast to your standard nullable integer
df_films_aux['movie_awards_score'] = (
    # (df_films_aux['oscar_wins'] * 10) +
    # (df_films_aux['oscar_nominations'] * 5) +
    (df_films_aux['total_wins'] * 10) +
    (df_films_aux['total_nominations'] * 5)
).astype('Int64')

# 2. Apply log10 to tame the massive outliers 
# (We temporarily cast to float for the numpy math, adding 1 to avoid log(0))
log_scores = np.log10(df_films_aux['movie_awards_score'].astype(float) + 1)

# 3. Min-Max normalize to a 0-100 scale, round to 2 decimals, and cast to Float64
min_log = log_scores.min()
max_log = log_scores.max()

df_films_aux['movie_awards_normalized'] = (
    ((log_scores - min_log) / (max_log - min_log)) * 100
).round(2).astype('Float64')

# Verify the output
cols_to_view = [
    'oscar_nominations', 'oscar_wins', 
    'total_nominations', 'total_wins', 
    'movie_awards_score', 'movie_awards_normalized'
]
print(df_films_aux[cols_to_view].head().to_string())

   oscar_nominations  oscar_wins  total_nominations  total_wins  movie_awards_score  movie_awards_normalized
0                  5           1                  9           4                  85                    49.54
1                  1           0                  2           1                  20                    33.86
2                  2           1                  2           1                  20                    33.86
3                  4           0                 13           7                 135                    54.64
4                  1           0                  2           0                  10                    26.67


In [35]:
# 1. Create the tie-breaker score (Wins as integers, Nominations as decimals)
composite_oscar_score = (
    df_films_aux['oscar_wins'].fillna(0) + 
    (df_films_aux['oscar_nominations'].fillna(0) * 0.01)
)

# 2. Rank the combined score
df_films_aux['oscar_wins_pct'] = (
    composite_oscar_score.rank(method='min', pct=True) * 100
).round(2).astype('Float64')

# Show results
cols_to_view = [
    'film_title', 'year',
    'oscar_nominations', 'oscar_wins', 
    'oscar_wins_pct', 'movie_awards_normalized'
]

print(df_films_aux[cols_to_view].head(10).to_string())
# print()
# print(df_films_aux[['oscar_wins_pct']].describe())

             film_title  year  oscar_nominations  oscar_wins  oscar_wins_pct  movie_awards_normalized
0      The Gay Divorcee  1934                  5           1           81.37                    52.48
1      She Loves Me Not  1934                  1           0           12.85                    32.13
2  Gold Diggers of 1935  1935                  2           1           73.66                    41.74
3               Top Hat  1935                  4           0           61.67                    52.86
4               Roberta  1935                  1           0           12.85                    28.84
5            Swing Time  1936                  2           1           73.66                    50.65
6         Born to Dance  1936                  2           0           44.75                    33.92
7   Pennies from Heaven  1936                  1           0           12.85                    26.05
8      Sing, Baby, Sing  1936                  1           0           12.85      

### Normalización Popularidad Films

In [40]:
# ==========================================
# Casting the data types
# ==========================================

# 1. Counts (Cast to nullable Int64)
# IMDB Votes: Force string, remove commas, coerce
df_films_aux['omdb_imdb_votes'] = pd.to_numeric(
    df_films_aux['omdb_imdb_votes'].astype(str).str.replace(',', ''), errors='coerce'
).astype('Int64')


# 2. Existing 0-100 Scores (Clean and cast DIRECTLY to nullable Float64)
# Rotten Tomatoes: Force string, remove '%', coerce
df_films_aux['omdb_rotten_tomatoes'] = pd.to_numeric(
    df_films_aux['omdb_rotten_tomatoes'].astype(str).str.replace('%', ''), errors='coerce'
).astype('Float64')

# Metascore: Coerce "N/A"
df_films_aux['omdb_metascore'] = pd.to_numeric(
    df_films_aux['omdb_metascore'], errors='coerce'
).astype('Float64')


# 3. 0-10 Scores (Clean, rescale to 0-100, round, and cast to nullable Float64)

# IMDB Rating: First, coerce "N/A" and cast to Float64
df_films_aux['omdb_imdb_rating'] = pd.to_numeric(
    df_films_aux['omdb_imdb_rating'], errors='coerce'
).astype('Float64')

# Only multiply by 10 if the column hasn't been scaled yet
if df_films_aux['omdb_imdb_rating'].max() <= 10:
    df_films_aux['omdb_imdb_rating'] = (df_films_aux['omdb_imdb_rating'] * 10).round(2)

# TMDB Vote Average: Cast to Float64
df_films_aux['tmdb_vote_average'] = df_films_aux['tmdb_vote_average'].astype('Float64')

# Only multiply by 10 if the column hasn't been scaled yet
if df_films_aux['tmdb_vote_average'].max() <= 10:
    df_films_aux['tmdb_vote_average'] = (df_films_aux['tmdb_vote_average'] * 10).round(2)
    
# Verify the changes
cols_to_view = [
    'tmdb_vote_average', 'tmdb_vote_count',
    'omdb_imdb_rating', 'omdb_imdb_votes', 
    'omdb_metascore', 'omdb_rotten_tomatoes'
]
print(df_films_aux[cols_to_view].dtypes)
print("\n", df_films_aux[cols_to_view].head().to_string())

tmdb_vote_average       Float64
tmdb_vote_count           int64
omdb_imdb_rating        Float64
omdb_imdb_votes           Int64
omdb_metascore          Float64
omdb_rotten_tomatoes    Float64
dtype: object

    tmdb_vote_average  tmdb_vote_count  omdb_imdb_rating  omdb_imdb_votes  omdb_metascore  omdb_rotten_tomatoes
0              68.91              147              73.0             9241            80.0                  93.0
1               60.0               11              57.0              227            <NA>                  <NA>
2              65.09               56              69.0             3123            <NA>                 100.0
3              72.53              331              77.0            22008            93.0                 100.0
4              70.82               49              70.0             3720            66.0                  86.0


In [42]:
# === 1. Audience Score (movie_popularity) ===

# === Data Quality Check: Nullify ghost votes ===
# If the rating is missing, force the vote count to 0 so it doesn't skew the denominator
df_films_aux.loc[df_films_aux['tmdb_vote_average'].isna(), 'tmdb_vote_count'] = 0
df_films_aux.loc[df_films_aux['omdb_imdb_rating'].isna(), 'omdb_imdb_votes'] = 0

# Calculate weighted values, filling nulls with 0 to prevent math errors
tmdb_weighted = (df_films_aux['tmdb_vote_average'] * df_films_aux['tmdb_vote_count']).fillna(0)
imdb_weighted = (df_films_aux['omdb_imdb_rating'] * df_films_aux['omdb_imdb_votes']).fillna(0)

# Calculate total votes
total_votes = df_films_aux['tmdb_vote_count'].fillna(0) + df_films_aux['omdb_imdb_votes'].fillna(0)

# Initialize the column
df_films_aux['movie_popularity'] = pd.NA

# Apply the weighted average only where total_votes > 0
mask_has_votes = total_votes > 0
df_films_aux.loc[mask_has_votes, 'movie_popularity'] = (
    (tmdb_weighted[mask_has_votes] + imdb_weighted[mask_has_votes]) / total_votes[mask_has_votes]
).round(2)

# Cast to the consistent nullable float type
df_films_aux['movie_popularity'] = df_films_aux['movie_popularity'].astype('Float64')


# === 2. Critics Score (movie_critics_score) ===

# mean(axis=1) automatically ignores NaNs. 
df_films_aux['movie_critics_score'] = df_films_aux[
    ['omdb_metascore', 'omdb_rotten_tomatoes']
].mean(axis=1).round(2).astype('Float64')

# Let's verify the new combined metrics
cols_to_view = ['movie_popularity', 'movie_critics_score', 'tmdb_vote_average', 'omdb_imdb_rating', 
                'omdb_metascore', 'omdb_rotten_tomatoes']
print(df_films_aux[cols_to_view].head().to_string())

   movie_popularity  movie_critics_score  tmdb_vote_average  omdb_imdb_rating  omdb_metascore  omdb_rotten_tomatoes
0             72.94                 86.5              68.91              73.0            80.0                  93.0
1             57.14                 <NA>               60.0              57.0            <NA>                  <NA>
2             68.93                100.0              65.09              69.0            <NA>                 100.0
3             76.93                 96.5              72.53              77.0            93.0                 100.0
4             70.01                 76.0              70.82              70.0            66.0                  86.0


### Limpieza de tabla de Artistas

In [43]:
print("=== 1. IDENTIFYING THE DUPLICATES ===")
# keep=False tells Pandas to show us ALL rows involved in the duplication, not just the extras
# duplicates_mask = df_artists_aux.duplicated(subset=['artist_id'], keep=False)
# print(df_artists_aux[duplicates_mask].sort_values('artist_id'))

print("\n=== 2. CONSOLIDATING THE DATA ===")
# UPGRADE: Instead of typing out every single column manually, we use a Python 
# "dictionary comprehension" to tell Pandas to use 'first' (first non-null value) 
# for ALL columns in your table, except for 'artist_id' and 'roles'!
aggregation_logic = {col: 'first' for col in df_artists_aux.columns if col not in ['artist_id', 'roles']}

# For 'roles', we want to combine them just in case one row says 'performer' 
# and another says 'composer'. We use a quick lambda with a Python Set to ensure no duplicates.
aggregation_logic['roles'] = lambda x: ', '.join(sorted(set(', '.join(x.dropna()).split(', '))))

# We group by the ID, apply our logic, and reset the index so it remains a flat table
df_artists_clean = (
    df_artists_aux
    .groupby('artist_id', as_index=False)
    .agg(aggregation_logic)
)

# Reorder columns to match the original structure dynamically
df_artists_clean = df_artists_clean[df_artists_aux.columns]

print(df_artists_clean[[
    'artist_id', 'artist_name', 
    'roles', 'mb_artist_type'
    # 'artist_playcount', 'artist_listeners', 
    # 'mb_gender', 'mb_country', 'mb_begin_area', 'mb_disambiguation', 
    # 'mb_lifespan_begin', 'mb_lifespan_end', 'mb_aliases',
    # 'cm_artist_ranking', 'cm_artist_eas'
]].to_string())

=== 1. IDENTIFYING THE DUPLICATES ===

=== 2. CONSOLIDATING THE DATA ===
                              artist_id                         artist_name      roles mb_artist_type
0                            a_r_rahman                        A. R. Rahman  performer         Person
1                               aaliyah                             Aaliyah  performer         Person
2                     abraham_alexander                   Abraham Alexander  performer         Person
3                          abraham_ivan                        Abraham Ivan  performer           None
4                           adam_levine                         Adam Levine  performer           None
5                                 adele                               Adele  performer         Person
6                        adrian_quesada                      Adrian Quesada  performer         Person
7                             aerosmith                           Aerosmith  performer          Group
8        

### Tabla de Roles

In [44]:
# Credits Table

# PREPARE THE DATA (String to List)
columns_to_melt = ['original_artists', 'song_composers', 'song_lyricists']
df_credits = df_songs_aux[['song_id', 'year', 'song_title'] + columns_to_melt].copy()

# Convert the comma-separated strings into actual Python lists
for col in columns_to_melt:
    df_credits[col] = df_credits[col].astype(str).str.split(r',\s*')

# MELT (Unpivot the columns into rows)
# Take the 3 role columns and collapse them into two new columns: 'role' and 'person'
df_credits = pd.melt(
    df_credits,
    id_vars=['song_id', 'year', 'song_title'], # Columns to keep static
    value_vars=columns_to_melt,                # Columns to collapse
    var_name='role',
    value_name='person'
)

# EXPLODE (List items into separate rows)
df_credits = df_credits.explode('person')

# CLEANUP AND RENAME
# Drop any NaN values or strings that literally say 'nan' or 'None'
df_credits = df_credits.dropna(subset=['person'])
df_credits = df_credits[~df_credits['person'].isin(['nan', 'None', ''])]

# We wrap the Songwriters in a single category
role_mapping = {
    'original_artists': 'artist',
    'song_composers': 'composer', # 'composer',
    'song_lyricists': 'lyricist' # 'lyricist'
}
df_credits['role'] = df_credits['role'].map(role_mapping)

# Sort it so it looks pretty and chronological
df_credits = df_credits.sort_values(['year', 'song_title', 'role']).reset_index(drop=True)

print("=== THE NEW CREDITS TABLE ===")
print(df_credits)

=== THE NEW CREDITS TABLE ===
      song_id  year           song_title      role          person
0      193402  1934        Love In Bloom    artist     Bing Crosby
1      193402  1934        Love In Bloom  composer   Ralph Rainger
2      193402  1934        Love In Bloom  lyricist       Leo Robin
3      193401  1934      The Continental    artist    Fred Astaire
4      193401  1934      The Continental    artist   Ginger Rogers
...       ...   ...                  ...       ...             ...
2037   202504  2025  Sweet Dreams Of Joy  lyricist   Nicholas Pike
2038   202503  2025         Train Dreams    artist  Sierra Ferrell
2039   202503  2025         Train Dreams  composer       Nick Cave
2040   202503  2025         Train Dreams  composer   Bryce Dessner
2041   202503  2025         Train Dreams  lyricist       Nick Cave

[2042 rows x 5 columns]


### Normalización Métricas de Artistas

In [45]:
# Normalize Artists metrics

# 1. Cast string-objects to numeric Int64
artist_metrics = ['artist_playcount', 'artist_listeners']

for col in artist_metrics:
    df_artists_clean[col] = pd.to_numeric(df_artists_clean[col], errors='coerce').astype('Int64')

# 2. Replace any exact 0s with pd.NA (just like we did for songs)
df_artists_clean[artist_metrics] = df_artists_clean[artist_metrics].replace(0, pd.NA)

# Let's take a quick look at the distribution before normalizing
print("--- Raw Artist Metrics ---")
print(df_artists_clean[artist_metrics].describe())


# 3. Apply Percentile and Logarithmic Normalization
for col in artist_metrics:
    # A. Percentile to 0-100 Scale
    df_artists_clean[f'{col}_percentile'] = (
        df_artists_clean[col].rank(pct=True) * 100
    ).round(2)
    
    # B. Logarithmic to 0-100 Scale (Min-Max)
    # We drop NAs specifically for the log calculation to avoid warnings,
    # but Pandas maps the results back to the correct rows automatically.
    log_vals = np.log10(df_artists_clean[col].dropna().astype(float))
    log_min = log_vals.min()
    log_max = log_vals.max()
    
    df_artists_clean[f'{col}_normalized'] = (
        ((np.log10(df_artists_clean[col].astype(float)) - log_min) / (log_max - log_min)) * 100
    ).round(2)

print("\n--- Normalized Artist Metrics Preview ---")
cols_to_view = ['artist_name', 'artist_listeners', 'artist_listeners_normalized', 'artist_playcount_normalized']
print(df_artists_clean[cols_to_view].head(10).to_string())

--- Raw Artist Metrics ---
       artist_playcount  artist_listeners
count             591.0             591.0
mean    42854795.751269     933062.235195
std    152416016.766594    1433803.947009
min                 1.0               1.0
25%            166593.0           25600.5
50%           2255750.0          287069.0
75%          17693447.0         1203446.0
max        1622785028.0         8436184.0

--- Normalized Artist Metrics Preview ---
         artist_name  artist_listeners  artist_listeners_normalized  artist_playcount_normalized
0       A. R. Rahman            689682                        84.30                        78.80
1            Aaliyah           2060186                        91.16                        82.89
2  Abraham Alexander             30076                        64.66                        55.11
3       Abraham Ivan              <NA>                          NaN                          NaN
4        Adam Levine            360555                        80.23

In [46]:
# ---------------------------------------------------------
# 1. Normalizing Artist Equivalent Album Sales (EAS)
# ---------------------------------------------------------
col_artist_eas = 'cm_artist_eas'

# Fill missing artist sales with 0
artist_eas_filled = df_artists_clean[col_artist_eas].fillna(0).astype(float)

# Percentile (Updated with method='min')
df_artists_clean[f'{col_artist_eas}_percentile'] = (
    artist_eas_filled.rank(pct=True, method='min') * 100
).round(2)

# Logarithmic Min-Max (with Log Shift +1)
log_vals_artist_eas = np.log10(artist_eas_filled + 1)
log_min_artist_eas = log_vals_artist_eas.min()
log_max_artist_eas = log_vals_artist_eas.max()

df_artists_clean[f'{col_artist_eas}_normalized'] = (
    ((log_vals_artist_eas - log_min_artist_eas) / (log_max_artist_eas - log_min_artist_eas)) * 100
).round(2)

# ---------------------------------------------------------
# 2. Normalizing the Artist Ranking Column
# ---------------------------------------------------------
col_artist_rank = 'cm_artist_ranking'

# Find the worst (highest number) rank on the chart
worst_artist_rank = df_artists_clean[col_artist_rank].max()

# Fill unranked artists with worst_rank + 1
artist_rank_filled = df_artists_clean[col_artist_rank].fillna(worst_artist_rank + 1).astype(float)

# A. Inverted Percentile (Updated with method='min')
df_artists_clean[f'{col_artist_rank}_percentile'] = (
    artist_rank_filled.rank(pct=True, ascending=False, method='min') * 100
).round(2)

# B. Inverted Logarithmic Min-Max
log_vals_artist_rank = np.log10(artist_rank_filled)
log_min_artist_rank = log_vals_artist_rank.min()
log_max_artist_rank = log_vals_artist_rank.max()

df_artists_clean[f'{col_artist_rank}_normalized'] = (
    100 - (((log_vals_artist_rank - log_min_artist_rank) / (log_max_artist_rank - log_min_artist_rank)) * 100)
).round(2)

# View the results
cols_to_view = ['artist_name', # 'artist_listeners', 
                'cm_artist_eas_percentile', 'cm_artist_eas_normalized',
                'cm_artist_ranking_percentile', 'cm_artist_ranking_normalized']
print(df_artists_clean[cols_to_view].head(10).to_string())


# Drop Columns

# 1. Drop the logarithmic ranking columns from the Songs table
songs_cols_to_drop = ['am_ranking_normalized', 'cm_ranking_normalized']
df_songs_aux.drop(columns=songs_cols_to_drop, inplace=True, errors='ignore')

# 2. Drop the logarithmic ranking column from the Artists table
artists_cols_to_drop = ['cm_artist_ranking_normalized']
df_artists_clean.drop(columns=artists_cols_to_drop, inplace=True, errors='ignore')

# Let's verify what remains for our ranking columns
print()
print("--- Songs Table Rankings ---")
print([col for col in df_songs_aux.columns if 'ranking' in col])

print("\n--- Artists Table Rankings ---")
print([col for col in df_artists_clean.columns if 'ranking' in col])

         artist_name  cm_artist_eas_percentile  cm_artist_eas_normalized  cm_artist_ranking_percentile  cm_artist_ranking_normalized
0       A. R. Rahman                     85.71                     86.02                         85.71                         10.65
1            Aaliyah                     85.22                     83.67                         85.22                          6.46
2  Abraham Alexander                      0.17                      0.00                          0.17                          0.00
3       Abraham Ivan                      0.17                      0.00                          0.17                          0.00
4        Adam Levine                      0.17                      0.00                          0.17                          0.00
5              Adele                     95.02                     92.77                         95.02                         38.99
6     Adrian Quesada                      0.17                      0

### Columna Número de Autores

In [54]:
# display(df_songs_aux[['song_composers', 'song_lyricists']].head())

def count_unique_writers(row):
    # 1. Grab the strings, treating NaNs/nulls as empty strings
    composers = str(row['song_composers']) if pd.notna(row['song_composers']) else ""
    lyricists = str(row['song_lyricists']) if pd.notna(row['song_lyricists']) else ""
    
    # 2. Split by comma, strip whitespace, and ignore empty values
    comp_list = [name.strip() for name in composers.split(',') if name.strip()]
    lyric_list = [name.strip() for name in lyricists.split(',') if name.strip()]
    
    # 3. Combine both lists to a set to automatically remove duplicates
    unique_writers = set(comp_list + lyric_list)
    
    # 4. Total count
    return len(unique_writers)

# Apply the function row by row and save it to the new column
df_songs_aux['num_ind_songwriters'] = df_songs_aux.apply(count_unique_writers, axis=1)

# Display a quick sample to verify it worked (e.g., showing songs with the most writers)
print(df_songs_aux[['song_title', 'song_composers', 'song_lyricists', 'num_ind_songwriters']].sort_values(by='num_ind_songwriters', ascending=False).head().to_string())

               song_title                                                                                                  song_composers                                                                                                  song_lyricists  num_ind_songwriters
390  Accidentally In Love                     Adam Duritz, Charles Gillingham, Jim Bogios, David Immergluck, Matthew Malley, David Bryson                                                                                     Adam Duritz, Daniel Vickrey                    7
492                Golden                    EJAE, Mark Sonnenblick, Joong Gyu Kwak, Yu Han Lee, Hee Dong Nam, Jeong Hoon Seo, Teddy Park                    EJAE, Mark Sonnenblick, Joong Gyu Kwak, Yu Han Lee, Hee Dong Nam, Jeong Hoon Seo, Teddy Park                    7
445         See You Again  Charlie Puth, Andrew Cedar, Cameron Thomaz, Dann Hume, Joshua Karl Simon Hardy, Justin Franks, Phoebe Cockburn  Charlie Puth, Andrew Cedar, Cameron Thomaz, Dann

## MUESTRA DE MEDIDAS

In [47]:
target_song_ids = [193502, 194401, 195701, 196602, 197702, 198203, 199701, 200001, 201801, 202101]

# Filter the junction table
df_sample_facts = df_facts_raw[df_facts_raw['song_id'].isin(target_song_ids)]

# Merge with the songs table
df_sample_merged = df_sample_facts.merge(
    df_songs_aux, 
    on='song_id', 
    how='left'
)

# Merge with the films table
df_sample_final = df_sample_merged.merge(
    df_films_aux, 
    on='film_id', 
    how='left'
)

# Metrics and identifiers to peek at
inspection_columns = [
    # 'song_id',
    'year_x',
    'song_title_x',
    'film_title_y',
    'oscar_nominations', 'oscar_wins',    
    'recordings_metric_pct', 
    'streams_obs_max', 'streams_obs_normalized', 
    'lastfm_track_listeners', 'lastfm_track_listeners_normalized',
    'movie_popularity', 'movie_critics_score'
    'budget_adjusted_normalized', 'revenue_adjusted_normalized',
    'movie_awards_normalized'
]

# Extract the columns
df_inspection = df_sample_final.reindex(columns=inspection_columns).dropna(axis=1, how='all')

display(df_inspection)
# print(df_songs_aux.head())

,year_x,song_title_x,film_title_y,oscar_nominations,oscar_wins,recordings_metric_pct,streams_obs_max,lastfm_track_listeners,lastfm_track_listeners_normalized,movie_popularity,revenue_adjusted_normalized,movie_awards_normalized
0,1935,Cheek To Cheek,Top Hat,4,0,97.61,210100000,466316,89.58,76.93,70.89,54.64
1,1944,Swinging On A Star,Going My Way,10,7,85.87,14200000,63142,75.86,69.96,19.30,65.34
2,1957,All The Way,The Joker Is Wild,1,1,91.74,43200000,169744,82.65,71.03,65.52,33.86
3,1966,Alfie,Alfie,5,0,94.02,6000000,40157,72.75,69.94,NaN,60.53
4,1977,Nobody Does It Better,The Spy Who Loved Me,3,0,78.80,67700000,212252,84.18,69.96,88.50,53.34
5,1982,Eye Of The Tiger,Rocky III,1,0,90.65,1833600000,1852253,99.05,69.0,82.62,50.17
6,1997,My Heart Will Go On,Titanic,14,11,94.78,714970000,255451,85.45,79.98,98.97,86.15
7,2000,Things Have Changed,Wonder Boys,3,1,73.48,20790000,142710,81.45,71.96,69.62,70.1
8,2018,Shallow,A Star Is Born,8,1,66.74,3118711700,950176,94.47,75.97,84.63,88.77
9,2021,No Time To Die,No Time to Die,3,1,46.74,797166000,862252,93.80,73.01,88.03,78.44


## EXPORTACIÓN DATASETS

In [55]:
df_songs_clean = df_songs_aux[[
    'song_id', 'song_title', 'year', 'decade',
    'oscar_song_nominee', 'oscar_song_win',
    'grammy_song_nominee', 'grammy_song_win', 'grammy_record_nominee',
    'song_nominations', 'song_wins', 'song_major_awards', 
    'awards_combined', 'awards_combined_percent',
    'grammy_record_win', 'song_composers', 'song_lyricists',
    'original_artists', 'other_artists', # 'streams_original', 'streams_others',
    'spotify_release_date', # 'spotify_duration_ms',
    'spotify_track_title', 'spotify_artists', # 'spotify_album_title',
    'lastfm_track_title', 'lastfm_artist_name', # 'lastfm_album_title',
    'lastfm_track_playcount', 'lastfm_track_listeners',
    'wiki_summary', 'mb_work_title', 'mb_composers', # 'lastfm_tags', 
    'mb_lyricists', 'mb_total_recordings', 'mb_original_recordings_count',
    'mb_cover_recordings_count', 'mb_covers_performers_count', 'mb_covers_performers_top',
    'mb_covers_earliest_year', 'mb_covers_latest_year',
    'mb_covers_earliest_performer', 'mb_covers_latest_performer', 
    'am_ranking', 'am_song_artist', 'am_song_year', 
    'cm_ranking', 'cm_song_artist', 'cm_song_eas', # 'match_key_am', 'match_key_cm', 
    'streams_obs_max', 'streams_percentile', 'streams_normalized', # 'streams_obs_normalized',
    'lastfm_track_playcount_percentile', 'lastfm_track_playcount_normalized', 
    'lastfm_track_listeners_percentile','lastfm_track_listeners_normalized', 
    'cm_song_eas_percentile', 'cm_song_eas_normalized', 
    'am_ranking_percentile', 'cm_ranking_percentile', 
    'mb_original_recordings_pct', 'mb_cover_recordings_pct', 
    'weighted_recordings_raw', 'recordings_metric_pct', 'num_ind_songwriters'
    ]].copy()

df_films_clean = df_films_aux[[
    'film_id', 'film_title', 'year', 'decade', 
    'film_directors', 'film_composers',
    'oscar_nominations', 'oscar_wins', 'oscar_wins_pct',
    'total_nominations', 'total_wins',
    'movie_awards_score', 'movie_awards_normalized', 
    'movie_popularity', 'movie_critics_score',
    'tmdb_official_title', 'tmdb_original_title', 'tmdb_release_date', 'tmdb_runtime',
    'tmdb_budget', 'tmdb_revenue', 'tmdb_vote_average', 'tmdb_vote_count',
    'tmdb_original_language', 'tmdb_origin_countries', 
    'tmdb_actors', 'tmdb_directors', 'tmdb_composers', 'tmdb_production_companies',
    'tmdb_genres', 'tmdb_tagline', 
    'omdb_title', 'omdb_awards',
    'omdb_imdb_rating', 'omdb_imdb_votes', 'omdb_metascore', 'omdb_rotten_tomatoes', 
    'budget_adjusted', 'revenue_adjusted', 'cpi_multiplier',
    'roi_percent', 'commercial_success_factor',
    'budget_adjusted_percentile', 'budget_adjusted_normalized',
    'revenue_adjusted_percentile', 'revenue_adjusted_normalized',
    'roi_percent_percentile', 'roi_percent_normalized',
    'commercial_success_factor_percentile', 'commercial_success_factor_normalized'
    ]].copy()

df_facts_clean = df_facts_raw.copy()
df_roles_clean = df_credits.copy()

In [56]:
df_songs_clean.to_pickle('data/clean_songs_df.pkl')
df_films_clean.to_pickle('data/clean_films_df.pkl')
df_facts_raw.to_pickle('data/clean_facts_df.pkl')
df_artists_clean.to_pickle('data/clean_artists_df.pkl')
df_roles_clean.to_pickle('data/clean_roles_df.pkl')

print("Data saved successfully!")

Data saved successfully!
